**← Previous:** [LGD Model Development](05_LGD_model_development.ipynb)  
**→ Next:** [Capital](07_capital.ipynb)

```text
Pipeline
├── 1. Setup & Import
├── 2. Data Loading
├── 3. Target Definition
├── 4. Split
├── 5. Model Configuration
├── 6. Training
├── 7. Prediction & Evaluation
└── 8. Export Pipeline
```

# Pipeline 

#### 1. Setup & Import

In [1]:
# Init work dir

from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
# Standard modules

import numpy as np
import pandas as pd

# Framework

from src.feature_engineering import create_target_def12
from src.modelling import fit_lgd, transform_lgd
from src.preprocessing import input_load, time_split

# Dataset specific settings

from src import settings_dataset_unsecured as settings

# Configuration

from sklearn import set_config; set_config(transform_output="pandas") # set sklearn output: pandas


#### 2. Data Loading

In [3]:
# load input file (DATASET SPECIFIC)

full_dataset = input_load(2019, 2020, column_date=settings.COLUMN_DATE, column_year=settings.COLUMN_YEAR)

#### 3. Target Definition

In [4]:
# CREATE TARGET VARIABLE PD

target_exists = settings.COLUMN_TARGET in full_dataset.columns
if not target_exists:
    full_dataset, dr_summary = create_target_def12(
        full_dataset,
        column_date=settings.COLUMN_DATE,
        column_target_date=settings.COLUMN_TARGET_DATE,
        column_target=settings.COLUMN_TARGET,
        column_year=settings.COLUMN_YEAR,
        verbose=True
    )

# CHECK TARGET VARIABLE LGD

target_lgd = "LossGivenDefault"

target_exists = target_lgd in full_dataset.columns
if not target_exists:
    print("LGD target variable not found")

LoanDate range:     2019-01-01 00:00:00 2020-12-31 00:00:00
+12 months:         2020-01-01 00:00:00 2021-12-31 00:00:00
DefaultDate range:  2019-04-26 00:00:00 2023-10-13 00:00:00


Year,num_loans,num_defaults,default_rate
2019,28487,4879,0.171271
2020,19959,2294,0.114936


#### 4. Split

In [5]:
# TRAIN/TEST SPLIT

windows = {"train": (2019, 2019), "test":  (2020, 2020)}

splits = time_split(full_dataset, settings.COLUMN_YEAR, windows, target_lgd)

df_trai = splits["train"]["df"]
df_test = splits["test"]["df"]


SPLIT SUMMARY

TRAIN
Years        : 2019 - 2019
Shape (X, y) : (28487, 112), (28487,)

Year distribution
Year
2019    28487
Name: count, dtype: int64

Target mean : 0.1633
------------------------------------------------------------

TEST
Years        : 2020 - 2020
Shape (X, y) : (19959, 112), (19959,)

Year distribution
Year
2020    19959
Name: count, dtype: int64

Target mean : 0.1365
------------------------------------------------------------


#### 5. Model Configuration

In [6]:
# model configuration

method = "stochastic"

#### 6. Training

In [7]:
# training

lgd_model = fit_lgd(df_trai, method=method, one_variable=None, verbose=True)

LGD estimation method: stochastic
Zero LGD probability:     0.6711
Positive LGD probability: 0.3289

Positive LGD distribution:
  Empirical mean:     0.4967
  Beta mean:          0.4972
  Empirical variance: 0.006996
  Beta variance:      0.007007

Quantile | Empirical | Beta
0.10 | 0.3902 | 0.3889
0.25 | 0.4291 | 0.4397
0.50 | 0.4941 | 0.4971
0.75 | 0.5582 | 0.5547
0.90 | 0.6040 | 0.6055


#### 7. Prediction & Evaluation

##### Validation (dev model replication)

In [8]:
# validate distribution parameters

print('PARAMETERS:')
print(' pi_zero:', lgd_model["model"]["pi_zero"])
print(' alpha  :', lgd_model["model"]["alpha"])
print(' beta   :', lgd_model["model"]["beta"])

PARAMETERS:
 pi_zero: 0.6711482430582371
 alpha  : 17.24153965519875
 beta   : 17.43565417430375


##### Prediction

In [9]:
# prediction
y_pred_trai = transform_lgd(df_trai, lgd_model)
y_pred_test = transform_lgd(df_test, lgd_model)

#### 8. Export Model

In [ ]:
## export trained pipeline

import joblib
joblib.dump(lgd_model, "../models/PIPELINE_LGD_vXXX_xxxxx.pkl")

['../models/PIPELINE_LGD_vXXX_stochastic.pkl']